# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [1]:
def build_kernel(data_set, kernel_type='linear', d=2):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        kernel = np.dot(data_set, data_set.T) ** d
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [2]:
def choose_set_for_label(data_set, label):
    train_data_set, test_data_set, train_labels, test_labels = train_test_split(
        data_set, labels, test_size=0.3, random_state=15)
    
    train_labels = np.where(train_labels == label, 1, -1)
    test_labels = np.where(test_labels == label, 1, -1)
    
    return train_data_set, test_data_set, train_labels, test_labels

In [3]:
def get_labels_count(data_set):
    return len(np.unique(data_set))

In [4]:
## Use the code that we have implemented earlier:

In [5]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    objects_count = len(train_labels)
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

# def build_kernel(data_set, kernel_type='linear'):
#     kernel = np.dot(data_set, data_set.T)
#     if kernel_type == 'rbf':
#         sigma = 1.0
#         objects_count = len(data_set)
#         b = np.ones((len(data_set), 1))
#         kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
#                          + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
#         kernel = np.exp(kernel / (2. * sigma ** 2))
#     return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [6]:
# modify this part 
from sklearn.datasets import load_iris
import numpy as np
from sklearn.model_selection import train_test_split
import cvxopt

iris = load_iris()
X, y = iris.data, iris.target
data_set = iris.data
labels = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=15)

from sklearn.metrics import accuracy_score

for label in range(get_labels_count(labels)):
    train_data_set, test_data_set, train_labels, test_labels = choose_set_for_label(data_set, label)
    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(train_data_set, train_labels, kernel_type='poly')
    predicted = classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id)
    predicted = list(predicted.astype(int))
    
    
    print(accuracy_score(predicted, test_labels))

     pcost       dcost       gap    pres   dres
 0: -1.3094e+01 -3.8051e+03  1e+04  5e-01  1e-11
 1: -5.8885e-01 -4.5834e+02  7e+02  3e-02  9e-12
 2:  2.2553e-02 -1.4526e+01  2e+01  8e-04  1e-12
 3:  3.5564e-02 -1.2440e+00  2e+00  6e-05  1e-13
 4:  2.1661e-02 -3.8780e-02  7e-02  5e-07  2e-14
 5:  2.7100e-03 -5.1096e-03  8e-03  2e-16  4e-15
 6:  2.1968e-05 -1.0681e-03  1e-03  2e-16  2e-15
 7: -2.4977e-04 -4.7672e-04  2e-04  2e-16  6e-16
 8: -3.8023e-04 -4.9845e-04  1e-04  2e-16  6e-16
 9: -4.3300e-04 -4.4033e-04  7e-06  2e-16  6e-16
10: -4.3730e-04 -4.3738e-04  8e-08  2e-16  6e-16
Optimal solution found.
1.0
     pcost       dcost       gap    pres   dres
 0: -1.4343e+01 -4.2051e+03  1e+04  5e-01  1e-11
 1: -1.2247e+00 -5.4599e+02  8e+02  3e-02  1e-11
 2:  1.4727e-02 -1.6060e+01  2e+01  7e-04  2e-12
 3:  5.1401e-02 -9.5812e-01  1e+00  4e-05  1e-13
 4:  1.9804e-02 -3.8608e-02  6e-02  2e-16  2e-14
 5:  2.1742e-03 -4.2745e-03  6e-03  2e-16  5e-15
 6: -3.0340e-05 -9.2474e-04  9e-04  2e-16  